# IOQM Phase 0: Colab runner

Runs the Phase 0 GPU steps from `docs/plan.md`:

1. **Check the GPU**: is the A100 40GB or 80GB?
2. **Time every registered model.** The timings decide the Phase 2 model list and seed count.
3. **Smoke-test generation** on a few prompts, including a resume after an interruption.
4. **Re-score a 2023 run** with the fixed metrics, so old and new numbers can be compared.

Results go to Google Drive under `ioqm_data/`. Every step can be resumed: after a disconnect, re-run from the top and finished work is skipped.

Runtime: **Runtime → Change runtime type → A100 GPU**.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/ioqm_data'
import os; os.makedirs(DATA, exist_ok=True)

**Getting the code.** Pick one option:
- `github`: clone the `phase0-harness` branch. This only works once the branch has been pushed.
- `drive`: copy the repo folder to Drive first, e.g. `MyDrive/ioqm`, and set `DRIVE_REPO`.

In [ ]:
REPO_SOURCE = 'github'                      # 'github' or 'drive'
DRIVE_REPO = '/content/drive/MyDrive/ioqm'  # used only when REPO_SOURCE == 'drive'

import os
if REPO_SOURCE == 'github':
    if not os.path.exists('/content/ioqm'):
        !git clone -b phase0-harness https://github.com/semajyllek/ioqm.git /content/ioqm
    else:
        !cd /content/ioqm && git pull
else:
    !rm -rf /content/ioqm && cp -r "$DRIVE_REPO" /content/ioqm
%cd /content/ioqm

In [ ]:
# New pipelines (Flux2Klein, QwenImage, ZImage) need a recent diffusers.
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf
# Counter deps: ultralytics (YOLOv8x). Mask2Former/DETR come with transformers.
!pip install -q -r requirements.txt

In [ ]:
# Quick check that the harness imports and its tests pass (no GPU needed).
!pip install -q pytest
!python -m pytest tests -q

**Hugging Face login.** Needed for gated models: SD3.5 medium/large, FLUX.1-dev and FLUX.2-klein-9B.
Open each model page on huggingface.co and accept its license first, otherwise that model shows up as `error` in step 1.

In [ ]:
from huggingface_hub import login
login()

## 1. Time every registered model

This loads each model, makes 1 warm-up image and then 5 timed images, and records:
- seconds per image
- peak GPU memory
- which offload mode was needed
- projected GPU-hours for the ~12k-image evaluation subset

On out-of-memory it retries with CPU offload. Results are appended to Drive one model at a time, and models already timed are skipped.

Model weights total roughly 150GB+. Colab's local disk can fill up, so the next cell deletes each model's Hugging Face cache once it has been timed.

In [ ]:
import json, os, shutil, subprocess
from ioqm.generate import REGISTRY

SPEED = f'{DATA}/speed.jsonl'
# Cheapest first. Edit to taste.
MODELS = ['sd15', 'sdxl_base', 'z_image_turbo', 'flux2_klein_4b', 'sd35_medium', 'z_image',
          'flux2_klein_9b', 'sd35_large', 'flux1_dev', 'qwen_image', 'sdxl_base_refiner']

def timed_models():
    if not os.path.exists(SPEED):
        return set()
    return {json.loads(l)['model'] for l in open(SPEED) if l.strip()}

HF_CACHE = os.path.expanduser('~/.cache/huggingface/hub')
for m in MODELS:
    if m in timed_models():
        print(f'skip {m} (already timed)'); continue
    subprocess.run(['python', 'scripts/benchmark_speed.py', '--models', m, '--out', SPEED])
    # free local disk: delete this model's weights from the HF cache
    repo_dir = os.path.join(HF_CACHE, 'models--' + REGISTRY[m].hf_id.replace('/', '--'))
    shutil.rmtree(repo_dir, ignore_errors=True)

In [ ]:
import pandas as pd
rows = [json.loads(l) for l in open(SPEED) if l.strip()]
df = pd.DataFrame(rows)
cols = [c for c in ['model','status','offload','s_per_img_median','peak_mem_gb','projected_gpu_hours_12k','load_s','error'] if c in df]
df = df[cols].sort_values('s_per_img_median', na_position='last')
print('GPU:', rows[-1]['versions'].get('gpu'), rows[-1]['versions'].get('gpu_mem_gb'), 'GB')
print('Total projected GPU-hours (ok models, 12k images each):',
      round(df.loc[df.status == 'ok', 'projected_gpu_hours_12k'].sum(), 1))
df

## 2. Smoke-test generation and resuming

Generates 8 eval prompts × 2 seeds with one fast model, then runs again to confirm nothing is regenerated. Look at a few images to sanity-check the prompts.

In [ ]:
SMOKE_MODEL = 'z_image_turbo'
SMOKE_DIR = f'{DATA}/runs/smoke/{SMOKE_MODEL}'
!python scripts/generate.py --model $SMOKE_MODEL --prompts prompts/v3/eval_subset.jsonl --seeds 0 1 --max_images 16 --out_dir $SMOKE_DIR
# second run: should report "16 done, 0 to generate"
!python scripts/generate.py --model $SMOKE_MODEL --prompts prompts/v3/eval_subset.jsonl --seeds 0 1 --max_images 16 --out_dir $SMOKE_DIR

In [ ]:
from IPython.display import display
from PIL import Image
from ioqm.generate import read_manifest
for r in read_manifest(SMOKE_DIR)[:6]:
    print(r['prompt'], '| split:', r['split'], '| seed:', r['seed'], f"| {r['gen_seconds']}s")
    display(Image.open(f"{SMOKE_DIR}/images/{r['image']}").resize((384, 384)))

## 2b. First steering experiment: best-of-N with a verifier counter

1. Generate the 480-prompt pilot (`prompts/v3/pilot.jsonl`) with **16 seeds**, which is 7,680 images.
2. Detect every image with two counters, YOLOv8x and Mask2Former. Raw detections are stored, so thresholds can change later.
3. For k = 1, 2, 4, 8, 16, one counter (the **verifier**) picks the best of k images, and the **other** counter (the **evaluator**) scores the pick. Both role assignments are reported.

Check the timing table first: 7,680 images × seconds per image. At ~1.5 s per image that's about 3 GPU-hours. Use 8 seeds to halve it.

In [ ]:
BON_MODEL = 'z_image_turbo'        # pick a fast model from the timing table
N_SEEDS = 16
BON_RUN = f'{DATA}/runs/bon_pilot/{BON_MODEL}'
SEEDS = ' '.join(str(s) for s in range(N_SEEDS))
!python scripts/generate.py --model $BON_MODEL --prompts prompts/v3/pilot.jsonl --seeds $SEEDS --out_dir $BON_RUN

In [ ]:
# Detection is resumable. Each counter stores all detections with score >= 0.05.
!python scripts/score.py --run_dir $BON_RUN --detector yolov8x
!python scripts/score.py --run_dir $BON_RUN --detector mask2former

In [ ]:
KS = ' '.join(str(k) for k in (1, 2, 4, 8, 16) if k <= N_SEEDS)
!python scripts/best_of_n.py --run_dir $BON_RUN --counters yolov8x mask2former --ks $KS --by n_types

**How to read it.**
- **`selected (eval)`** is the result: count accuracy of the verifier's pick, judged by the other counter.
- **`gain vs k=1`** is the improvement over a single sample, with a paired 95% CI.
- **`verifier self`** is how good the verifier *thinks* its pick is. A big gap above `selected` means the verifier is being gamed.
- **`oracle/pass@k`** is the ceiling a perfect verifier would reach.
- **`[SELF]`** blocks, where verifier and evaluator are the same counter, are not results.

If the two role assignments (YOLO→Mask2Former and Mask2Former→YOLO) disagree a lot, the counters don't agree on these images. That's the Phase 1 instrument problem, and checking against the human labels decides which one to trust.

## 3. Full evaluation-subset generation (Phase 2, run when ready)

This is the ~3k prompts × 4 seeds run, about 12k images per model. Check the timing table first. Re-run the cell after a disconnect to resume.

In [ ]:
RUN_MODEL = 'z_image_turbo'   # change per model
SEEDS = '0 1 2 3'
!python scripts/generate.py --model $RUN_MODEL --prompts prompts/v3/eval_subset.jsonl --seeds $SEEDS --out_dir $DATA/runs/v3_eval/$RUN_MODEL

## 4. Re-score a 2023 run with the fixed metrics

This wraps a 2023 image folder as a run (the images are not copied), detects with YOLOv8x, and prints the new metrics by stratum. It is the before/after for the scoring bugs: compare it with the old `yolo_scores` files.

In [ ]:
LEGACY_FOLDER = f'{DATA}/generated_images_v2/stable-diffusion-v1-5_images'
LEGACY_RUN = f'{DATA}/runs/legacy_v2/sd15'
!python scripts/import_legacy_run.py --img_folder "$LEGACY_FOLDER" --model sd15 --dataset v2 --out_dir "$LEGACY_RUN"
!python scripts/score.py --run_dir "$LEGACY_RUN" --detector yolov8x

In [ ]:
# Re-score at other thresholds without re-running the detector
for t in (0.3, 0.7):
    !python scripts/score.py --run_dir "$LEGACY_RUN" --detector yolov8x --skip_detect --threshold $t